In [13]:
import pandas as pd
import numpy as np
import scipy as sp

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mpl_toolkits as mplot3d

import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from scipy.stats.mstats import winsorize


pd.set_option("display.max_columns", None)
pd.reset_option("display.max_rows")
pd.set_option('future.no_silent_downcasting', True)

from utils.categorical_qcut import categorical_qcut

from tqdm import tqdm
tqdm.pandas()

# Data Scoring

In [14]:
scenarios_answers_dict = {
    'M25-1': 'Try calling a close friend or hostel mate for immediate help, and together ensure he gets urgent medical care.',
    'M25-2': 'Use allergy medication if available and call for emergency help.',
    'M25-3': 'Call for the emergency',
    'M25-4': 'Clean the wound carefully, press to control the bleeding, cover it, and arrange to see a doctor right away.',
    'M25-5': 'Stop, sit upright, slow his breathing, and ask for help.',
    'M25-6': 'Try calling or messaging the bank’s helpline right away to report the situation and at the same time use the app to freeze or block his card if possible.',
    'M25-7': 'Hang up the call without giving any information, then look up and contact the official police station independently to check if the demand was real.',
    'M25-8': 'Call the bank directly using a number on their official website and ask if the loan email is genuine, without clicking the link.',
    'M25-9': 'Research the scheme independently using RBI or SEBI advisories.',
    'M25-10': 'Check his payment/card records separately and contact his provider if needed.',
    'M25-11': 'Ignore the link, verify status in-app, and enable security features.',
    'M25-12': 'Block UPI and logout from accounts immediately.',
    'M25-13': 'Refuse the caller, then disconnect and reach out to the bank based on the phone number on the debit card.',
    'M25-14': 'Change the account password, look through security settings, and enable additional protection like two factor login.',
    'M25-15': 'Save evidence, secure all accounts, seek support, and prepare a report.',
    'M25-16': 'Take screenshots of the posts and messages, and then reach out to a trusted authority figure or counsellor who can step in to support him.',
    'M25-17': 'Confront the friend calmly to clear things up, and if it keeps happening, get hostel staff involved.',
    'M25-18': 'Firmly refuse and step away from the situation if necessary, seeking help if required.',
    'M25-19': 'Keep the messages as evidence and reach out to a student helpline or official channel, even if he’s nervous.',
    'M25-20': 'Talk honestly about his feelings with a supportive friend, mentor, or counselor.',
    
    
    'M59-1': 'Take an aspirin to reduce the risk of heart trouble, then call emergency services.',
    'M59-2': 'Use the allergy medicine he carries with him—like an antihistamine—and at the same time ask someone nearby to get emergency help.',
    'M59-3': 'Grab his phone and dial emergency services straight away, telling them what’s happening even if his words come out unclear, and stay put until help arrives.',
    'M59-4': 'Quickly grab a cloth or towel to press firmly on the wound, raise his hand higher to slow the bleeding, and call for urgent help.',
    'M59-5': 'Sit down if possible, mention to someone nearby that he’s feeling unwell, and ask for anything sweet or sugary he can have while he waits for help.',

    'M59-6': 'Use his banking app or the helpline to stop further transactions right away and then check what else needs to be reported.',
    'M59-7': 'Hang up and independently verify via government channels.',
    'M59-8': 'Check his credit report with an official bureau and contact the bank.',
    'M59-9': 'Take some time to find out more about the start-up on his own, checking what’s publicly available, before choosing whether to invest.',
    'M59-10': 'Check his KYC status using the bank’s app or by logging in the usual way, without using the link in the email.',
    
    'M59-11': "Open his bank’s app or log in through his usual method to see if there are any actual alerts about his account, rather than using the link in the message.",
    'M59-12': 'Go into his account settings to change his password, add two-factor authentication, and make sure nobody else can access his email.',
    'M59-13': 'Tell the caller he’ll think about it and first check with his bank the usual way, instead of installing anything right away.',
    'M59-14': 'Regain access through the account recovery process and inform contacts.',
    'M59-15': 'Save any evidence of what happened, review and tighten the security on his accounts, and look for help reporting the breach.',
    
    'M59-16': 'Arrange to talk privately with his supervisor or someone in HR, and respectfully ask about the changes and exclusions to understand what’s behind them.',
    'M59-17': 'Save the posts and reach out to a society committee member for help.',
    'M59-18': 'Contact a trusted friend or relative and explore together possible ways to reconnect.',
    'M59-19': 'Clearly state his limitations and seek shared responsibility from family members.',
    'M59-20': 'Approach his friend in a kind and genuine manner to ask about the situation.',

    
    'M60-1': 'Take aspirin and seek help immediately.',
    'M60-2': 'Take his allergy medicine if he has it, and ask someone nearby to get urgent medical help instead of waiting to see if it improves.',
    'M60-3': 'Try to call for medical help right away and explain what’s happening as best he can, rather than waiting alone.',
    'M60-4': 'Apply pressure and elevate the hand, then get help.',
    'M60-5': 'Sit down right away, ask someone nearby if they have anything sweet he can eat, and mention that he’s not feeling well so he can get help if needed.',

    'M60-6': 'Call his bank’s helpline right away to talk about what’s happening and find out how to stop further transactions.',
    'M60-7': 'Hang up and call the Income Tax Department on the number he normally uses, just to make sure the request from the caller is actually genuine.',
    'M60-8': 'Contact the concerned bank and credit bureau to report the issue.',
    'M60-9': 'Decline and independently research the investment scheme.',
    'M60-10': 'Get in touch with his bank using the official website or stop by in person to ask if the alert is genuine before taking any action.',

    'M60-11': 'Log in using the bank’s app he usually uses, or wait to ask someone at his branch about the alert.',
    'M60-12': 'Change his password and set up two-factor authentication.',
    'M60-13': 'Refuse, disconnect, and verify with his bank directly.',
    'M60-14': 'Try to recover his account using the recovery process, and let his contacts know so they’re aware something unusual is happening.',
    'M60-15': 'Gather any evidence of what’s happened, review the security of his accounts, and ask someone he trusts to help him report what’s going on.',

    'M60-16': 'Contact a trusted friend or community group to reconnect socially.',
    'M60-17': 'Speak one-on-one with a trusted neighbor or a society official to get their honest perspective and, if needed, clear up any misunderstandings in private.',
    'M60-18': 'Open a conversation with a family member he trusts to seek understanding and resolution.',
    'M60-19': "Tell his family honestly that he needs help and identify specific support he’d appreciate.",
    'M60-20': 'Ask a friendly neighbor in a polite, non-confrontational way if he missed an invite.',


    'W25-1': 'Call for help from hostel staff or friends, and get to a campus clinic or similar immediately.',
    'W25-2': 'Use her allergy medication (if available) and seek emergency help immediately.',
    'W25-3': 'Call for the emergency',
    'W25-4': 'Clean the wound, press to stop the bleeding, and get someone to take her to a clinic.',
    'W25-5': 'Sit down and focus on slowing her breathing, then look for a way to get help, rather than continuing her run.',

    'W25-6': 'Use her banking app or call the helpline right away to block her card and inform the bank about what’s happened.',
    'W25-7': 'Hang up, then verify the demand with official government contacts.',
    'W25-8': 'Contact her bank and the credit bureau as soon as possible to report the problem and ask what steps she should take.',
    'W25-9': 'Look up details about the scheme using official sites or check what financial regulators say before deciding what to do next.',
    'W25-10': 'Do not open the PDF; verify independently using her bank app or statement.',

    'W25-11': "Skip clicking the link, open her Instagram app directly to check for any issues, and consider updating her account’s security settings.",
    'W25-12': 'Use her banking app to block UPI transactions right away and let her bank know what’s happened.',
    'W25-13': 'Say no to sharing her OTP, end the call, and check with the actual wallet app’s helpline to see if there really is a problem.',
    'W25-14': 'Go into her email account, update her password and add some extra security options.',
    'W25-15': 'Save evidence and alert someone she trusts while preparing to report.',

    'W25-16': "Save screenshots as evidence and approach a counselor or a trusted adult for help.",
    'W25-17': 'Have a calm one-on-one conversation with her roommate or talk to the hostel warden.',
    'W25-18': 'Say clearly that she doesn’t want to drink and, if things feel too uncomfortable, move away from the group for a bit.',
    'W25-19': 'Keep the messages as evidence and reach out to the student support office or someone at the university who can help.',
    'W25-20': 'Speak with a counselor or someone she trusts about feeling left out and work through her emotions.',




    'W59-1': 'Take an aspirin if it’s available, call for help, and let someone know about her symptoms.',
    'W59-2': 'Take her allergy medicine if she has it, and get someone to call for medical help right away.',
    'W59-3': 'Call emergency medical services at once.',
    'W59-4': 'Apply direct pressure, elevate hand and call for help.',
    'W59-5': 'Sit down, ask someone nearby for something sweet, and let them know she’s not feeling well.',

    'W59-6': 'Use her bank’s app or call to block the card as soon as possible and tell the bank about the charges.',
    'W59-7': 'Hang up and independently contact the Income Tax Department.',
    'W59-8': 'Check her credit report through official channels and alert her bank.',
    'W59-9': 'Look up information about the scheme herself and see what she can find from reliable sites before making a decision.',
    'W59-10': 'Use her bank’s app or visit the branch to see if her KYC is actually pending before responding to the email.',
    
    'W59-11': 'Skip the link and log in with her usual banking app or website to check if there’s a real issue.',
    'W59-12': 'Change her password and turn on two-factor authentication.',
    'W59-13': 'Refuse and verify with official IT support.',
    'W59-14': 'Change her password and secure her account.',
    'W59-15': 'Save evidence, secure accounts, and reach out for support while reporting.',

    'W59-16': 'Reach out directly to her supervisor or HR to discuss her concerns.',
    'W59-17': 'Save the hurtful messages as evidence and request intervention from building management.',
    'W59-18': 'Reach out to a trusted friend or counselor and share her feelings honestly.',
    'W59-19': 'Talk honestly and respectfully with her in-laws about boundaries and shared responsibilities.',
    'W59-20': 'Privately and calmly seek clarification, possibly requesting mediation.',


    'W60-1': 'Take an aspirin if she has one, call for help straight away, and let someone know what she’s feeling.',
    'W60-2': 'Stay still and shout for assistance.',
    'W60-3': 'Stop reading, call for help, and get to the hospital.',
    'W60-4': 'Move to a safe spot, let a nearby neighbor know she’s confused, and ask for help getting checked by a doctor.',
    'W60-5': 'Pause before taking the pills, call the pharmacy to explain what she’s noticed, and ask if there’s been any change to her prescription.',

    'W60-6': 'Call the bank’s customer service line right away to report the missing money and freeze her account.',
    'W60-7': 'Hang up, then call her bank using the number listed on her bank documents.',
    'W60-8': 'Contact her bank or credit service to ask about placing a fraud alert on her credit profile.',
    'W60-9': 'Decide to find out more about the chit fund through independent sources before making any choice.',
    'W60-10': 'Check her fixed deposit status by logging into her usual banking app or visiting her bank.',

    'W60-11': 'Verify with someone she trusts and ignore the suspicious message.',
    'W60-12': 'Ask a trusted family member or neighbor for help reading the instructions.',
    'W60-13': 'Refuse and independently verify with someone she knows.',
    'W60-14': 'Set the phone to airplane mode and seek help to check for malware or unwanted apps.',
    'W60-15': 'Change her password and secure the account.',
    
    'W60-16': 'Reach out to someone she trusts and share how she’s feeling.',
    'W60-17': 'Talk privately with someone she trusts in the group to clear things up or address what she’s heard.',
    'W60-18': 'Reflect for a while and then gently bring up her feelings with a family member.',
    'W60-19': 'Let her family know about her concerns and clearly request specific help where needed.',
    'W60-20': 'Gently ask a trusted neighbor if there was a reason she was left out.',


}

# Data Cleaning

In [16]:
# importing
import_path = f"../../data_cleaned/india/dynata_pilot 2/"
data = pd.read_csv(f"{import_path}/DynataScenarioPiloting_August 28.csv")

questions_df = data[:1]

data = data[2:].copy()

# removing samples from pilot 1
data["StartDate"] = pd.to_datetime(data["StartDate"], format = "%Y-%m-%d %H:%M:%S")
data = data.loc[data["StartDate"] >= "2025-08-27"].copy().reset_index(drop = True)

data["Q2"] = data["Q2"].astype(float)
raw_data = data.copy()

# scoring
print("Following questions have no correct answers in the demographic:")
for key, value in scenarios_answers_dict.items():
    data[key] = np.where(data[key].str.strip() == value.strip(), 1,
                         np.where(pd.isna(data[key].str.strip()), np.nan, 0
                                  )
                        )
    if data[key].sum() == 0:
        print(f"{key}: sum = {data[key].sum()}, count = {data[key].count()}")

# removing those that were filtered out due to demographic misalignment
data = data.copy()
data["score_sum"] = data.loc[:, list(scenarios_answers_dict.keys())].sum(axis = 1)
data = data.loc[ data["score_sum"]!= 0].copy()

# Removing preview responses
data = data.loc[ data["Status"] != "Survey Preview"].copy()

# <0.5 median time of completion filtered --> already done for this dataset
data["Duration (in seconds)"] = data["Duration (in seconds)"].astype("Int64")
median = data["Duration (in seconds)"].astype("Int64").median()
data = data.loc[ data["Duration (in seconds)"] >= median*0.5].copy()
print("\nPost removing <0.5 median time responses:", len(data["ResponseId"]))

output_col = ["ResponseId", "Q1", "Q2", "Q3", "Q4a"] + list(scenarios_answers_dict.keys())
output = data.loc[:, output_col].copy()
output.to_excel("dynata_pilot_cleaned_200825.xlsx")

Following questions have no correct answers in the demographic:

Post removing <0.5 median time responses: 75


In [18]:
scenarios_questions_dict = {}
for key in scenarios_answers_dict.keys():
    scenarios_questions_dict[key] = questions_df[key][0]

In [19]:
def sample_prep(data, sex, age):
    sex2 = "Male" if sex == "M" else "Female"
    if age == 25:
        age_upper = 25
        age_lower = 18
    elif age == 59:
        age_upper = 59
        age_lower = 26
    else:
        age_upper = 122
        age_lower = 60
        
    sample = data.dropna(subset = ["Q1", "Q2"]).loc[ (data["Q1"] == sex2) & ((data["Q2"] >= age_lower) & (data["Q2"] <= age_upper))].copy()
    return sample

def question_analysis(data, sex, age):

    sample = sample_prep(data, sex, age)
    if sex not in {"M", "W"}:
        raise ValueError("sex not applicable, input one of 'M' or 'W'")
    if age not in {25, 59, 60}:
        raise ValueError("age not applicable, input one of 25, 59, 60")

    col_prefix = f"{sex}{age}-"
    focal_columns = [x for x in sample.columns if x.startswith(col_prefix)]
    
    for col in focal_columns:
        print(f"{col}:{ sample[col].sum() / sample[col].count() }")


In [20]:
# Demographics
for sex in ["M", "W"]:
    for age in [25, 59, 60]:
        sex2 = "Male" if sex == "M" else "Female"
        if age == 25:
            age2 = "18-25"
        elif age == 59:
            age2 = "26-59"
        else:
            age2 = "60+"

        print(f"{sex2:<7} {age2:<7}: {len(sample_prep(data, sex, age))}")

Male    18-25  : 11
Male    26-59  : 15
Male    60+    : 13
Female  18-25  : 12
Female  26-59  : 17
Female  60+    : 7


# Score Averages for each demoghraphic

In [23]:
for sex in ["M", "W"]:
    for age in [25, 59, 60]:
        sex2 = "Male" if sex == "M" else "Female"
        if age == 25:
            age2 = "18-25"
        elif age == 59:
            age2 = "26-59"
        else:
            age2 = "60+"

        print(f"\n\n{sex2:<7} {age2:<7}: count: {len(sample_prep(data, sex, age))}")
        question_analysis(data, sex, age)



Male    18-25  : count: 25
M25-1:0.8
M25-2:0.76
M25-3:0.52
M25-4:0.84
M25-5:0.88
M25-6:0.76
M25-7:0.76
M25-8:0.76
M25-9:0.68
M25-10:0.52
M25-11:0.68
M25-12:0.48
M25-13:0.72
M25-14:0.88
M25-15:0.48
M25-16:0.76
M25-17:0.84
M25-18:0.64
M25-19:0.6
M25-20:0.6


Male    26-59  : count: 29
M59-1:0.9655172413793104
M59-2:1.0
M59-3:0.7586206896551724
M59-4:0.896551724137931
M59-5:0.9310344827586207
M59-6:0.896551724137931
M59-7:0.5862068965517241
M59-8:0.6551724137931034
M59-9:0.8620689655172413
M59-10:1.0
M59-11:0.8620689655172413
M59-12:0.8620689655172413
M59-13:0.896551724137931
M59-14:0.3103448275862069
M59-15:0.7241379310344828
M59-16:0.8275862068965517
M59-17:0.6551724137931034
M59-18:0.2413793103448276
M59-19:0.4482758620689655
M59-20:0.4827586206896552


Male    60+    : count: 13
M60-1:0.6153846153846154
M60-2:0.8461538461538461
M60-3:0.6923076923076923
M60-4:0.5384615384615384
M60-5:0.9230769230769231
M60-6:0.8461538461538461
M60-7:0.9230769230769231
M60-8:0.5384615384615384
M60-9:0

### < 0.305 Question analysis

In [36]:
def low_answer_rate_questions(data, sex, age, low_score_questions_set):

    sample = sample_prep(data, sex, age)
    if sex not in {"M", "W"}:
        raise ValueError("sex not applicable, input one of 'M' or 'W'")
    if age not in {25, 59, 60}:
        raise ValueError("age not applicable, input one of 25, 59, 60")

    col_prefix = f"{sex}{age}-"
    focal_columns = [x for x in sample.columns if x.startswith(col_prefix)]
    
    for col in focal_columns:
        if sample[col].sum() / sample[col].count() <= 0.305:
            low_score_questions_set.add(col)

In [37]:
low_scoring_questions = set()
for sex in ["M", "W"]:
    for age in [25, 59, 60]:
        sex2 = "Male" if sex == "M" else "Female"
        if age == 25:
            age2 = "18-25"
        elif age == 59:
            age2 = "26-59"
        else:
            age2 = "60+"

        low_answer_rate_questions(data, sex, age, low_scoring_questions)

In [38]:
low_scoring_questions

{'M59-18', 'M60-16', 'M60-20', 'W59-19', 'W59-20', 'W60-14'}

In [56]:
for col in low_scoring_questions:
    print(f"\n{raw_data[col].rename(f'{col}: {scenarios_questions_dict[col]}').value_counts().rename({scenarios_answers_dict[col]: f'Correct: {scenarios_answers_dict[col]}'})}")


W59-19: Jyoti, 33, is pressured by in-laws to assume full caregiving for an ill relative despite her work commitments. She feels overwhelmed but fears saying no will cause family friction.\r\n\r\nWhat is the best response Jyoti should do?
Talk openly and respectfully with her in-laws about boundaries and shared responsibilities.     12
Discuss her concerns with a close family member to seek their support.                          12
Suggest small changes to distribute tasks more evenly within the family.                         7
Quietly take on the role but seek help from her partner if it becomes too much.                  3
Name: count, dtype: int64

W59-20: Rupa, 48, sees her reputation damaged by a colleague’s false rumors at her cooperative society. She feels humiliated but is unsure whether to address it.\r\n\r\nWhat is the best response Rupa should do?
Privately approach the colleague to clarify concerns and seek a positive outcome.    15
Privately and calmly seek clarificatio